# Inductive evaluation: seen–unseen

This notebook presents the overall findings of the inductive generalization evaluation in terms of  **AUROC and F1 figures**, 
followed by the code snippet to run and re-generate the results.

The supplied aggregate tables can be used to redraw both figures.

Once you have the appropriate licences and have prepared the data using the
[data guide](../../data/README.md) and
[DataPreparation.ipynb](../../data/DataPreparation.ipynb), you can rerun the
evaluation and regenerate the figures using the functions in
[`src/dtpkg/inductive/`](../../src/dtpkg/inductive/).

## Paths and run options

Replace each `...` in the next cell with your own quoted path, for example
`DATA_DIR = "/path/to/private/data"`. `PROJECT_ROOT` is the checkout folder;
`DATA_DIR` follows the [data guide](../../data/README.md). In the fusion notebooks,
also supply `GRAPH_PATH`, the complete path to your GraphML file.
Then adjust the derived input and output paths as needed and rerun the cells.
The path cell intentionally requires completion before execution; no config-file
or environment-variable changes are needed. The default run uses only the public results.


In [ ]:
from pathlib import Path
import sys

# Replace each ... with your own quoted path or Path(...).
PROJECT_ROOT = ...  # DTP-KG checkout folder
DATA_DIR = ...  # Local data folder; see data/README.md
GRAPH_PATH = ...  # Complete path to the canonical GraphML network

# Check and normalize the paths supplied above.
if any(path is ... for path in (PROJECT_ROOT, DATA_DIR, GRAPH_PATH)):
    raise ValueError("Replace ... in PROJECT_ROOT, DATA_DIR, GRAPH_PATH with your local paths.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DATA_DIR = Path(DATA_DIR).expanduser().resolve()
GRAPH_PATH = Path(GRAPH_PATH).expanduser().resolve()

# Public aggregate inputs and regenerated figures.
EXPERIMENT = PROJECT_ROOT / "experiments" / "05_inductive_fusion"
TABLES = EXPERIMENT / "tables"
FIGURES = EXPERIMENT / "figures"
OUTPUT_DIR = FIGURES / "reproduced"

# Inputs and outputs for optional private preparation/training.
PRIVATE_PREPARED = DATA_DIR / "splits" / "inductive_seen_unseen_v1"
PRIVATE_RUN = EXPERIMENT / "results" / "inductive_fusion"

REPLOT = False
PREPARE_SPLITS = False
RUN_TRAINING = False


In [2]:
from IPython.display import display, Image
from dtpkg.inductive.reporting import load_report
from dtpkg.inductive.plot_paired import render_figures


## Drug holdouts and feature preparation

The study selects three overlapping holdouts using MeSH-depth × non-DDI-degree
strata. Each holds out 374 of 2,493 candidate drugs, leaving 2,119 development
drugs. The first three proposals satisfying the declared support rules were
accepted (selection seeds 30003, 30007, and 30008), without using model scores.

For seen–unseen evaluation, each of six depth categories is balanced between
positive and curated-negative pairs and must contain at least 20 pairs and five
distinct held-out contributors per class. Seen partners must occur in the first
supervised training epoch for all three training seeds (101, 202, and 303).
These are overlapping drug holdouts, not disjoint cross-validation folds.

The selected Intermediate MeSH scope is retained, but each split refits its
vocabulary, TF-IDF, and 128-component SVD on development drugs only. The global
embedding CSV supplies candidate row IDs; its fitted values are not reused.
Held-out drugs are transformed with the development-fitted representation.

All DDI edges are removed from both feature graphs. Held-out nodes are additionally
removed from the training graph. At inference, held-out drugs can use available
non-DDI connections. This evaluates drugs with available side information;
it does not represent drugs without known relations.

## Training protocol

Three training seeds per holdout produce **nine split–seed runs and 27 neural
model fits**. Each run fits fresh MeSH-only, topology-only, and fusion models.
Inner validation uses 15% of development pairs and selects checkpoints by
binary cross-entropy loss, with at most 100 epochs and patience 10. Adam uses
learning rate 0.001 and weight decay 0.00001.

Positive pairs are resampled each epoch at a 1:1 ratio to training negatives,
excluding held-out drugs and validation/test pairs. The MeSH scaler uses the
full planned training-pair schedule. Topology normalization uses training-drug
endpoints from that schedule, excluding held-out drugs. Fusion uses MeSH
modality dropout of 0.3 during training. Neural decisions use threshold 0.5.
MeSH-only updates in batches of 128; fusion and topology-only update per pair.
The methods therefore have different optimization budgets.

## Verify the aggregate results

The public report contains 189 split/seed/model/category rows. For each metric,
training seeds are averaged within a holdout before averaging the three holdouts
equally. The figure bars are **sample SD across those three holdout means**.
Nine split–seed runs are not nine independent holdouts.

Paired tests use three seed-averaged Fusion-minus-comparator differences, df = 2,
and the approximate overlap correction based on 374/2,119 held-out/development
drugs. Holm adjustment retains four overall comparisons (AUROC/F1 × two
comparators) and 60 depth comparisons (five metrics × six categories × two
comparators). Precision, recall, and accuracy remain in the depth family even
though the selected figures display only AUROC and F1.

In [3]:
report = load_report(TABLES)
summary = report["summary"]
assert len(report["overall"]) == 4
assert len(report["depth"]) == 60

model_names = {"baseline": "MeSH-only", "topo_only": "Topology-only", "fusion": "Fusion"}
overall = summary[summary.group_kind.eq("overall") & summary.metric.isin(["auc", "f1"])].copy()
overall["Mean ± holdout SD"] = overall.apply(
    lambda row: f"{row.estimate:.3f} ± {row.sd_across_holdouts:.3f}", axis=1)
values = overall.pivot(index="model", columns="metric", values="Mean ± holdout SD")
display(values.reindex(model_names).rename(index=model_names, columns={"auc": "AUROC", "f1": "F1"}))

overall_comparisons = report["overall"][["metric", "model_a", "model_b", "estimate", "p_raw", "p_holm"]].copy()
for column in ("model_a", "model_b"):
    overall_comparisons[column] = overall_comparisons[column].map(model_names)
display(overall_comparisons.round(4))

metric,AUROC,F1
model,,
MeSH-only,0.892 ± 0.019,0.802 ± 0.015
Topology-only,0.793 ± 0.038,0.711 ± 0.022
Fusion,0.937 ± 0.021,0.841 ± 0.021


,metric,model_a,model_b,estimate,p_raw,p_holm
0,auc,Fusion,MeSH-only,0.0451,0.0140,0.0281
1,auc,Fusion,Topology-only,0.1436,0.0083,0.0267
2,f1,Fusion,MeSH-only,0.0391,0.0432,0.0432
3,f1,Fusion,Topology-only,0.1307,0.0067,0.0267


## Seen–unseen AUROC

![Seen–unseen AUROC](figures/seen_unseen_auroc.png)

The panels show overall and MeSH-depth AUROC. Low, Mid, and Deep refer to each
drug's deepest annotation (levels 1–5, 6–7, and 8–10), not to the selected
representation scope. [Full caption](figures/seen_unseen_auroc_caption.txt).

## Optional: redraw both figures

Set `REPLOT = True` above. Plotting verifies `TABLES` and writes to `OUTPUT_DIR`
(by default, ignored `figures/reproduced/`), preserving the supplied references.

In [4]:
if REPLOT:
    reproduced = render_figures(TABLES, OUTPUT_DIR)
    for metric in ("auc", "f1"):
        display(Image(filename=str(reproduced[metric])))

## Optional: prepare and train with private inputs

Use the shared tables from [data preparation](../../data/DataPreparation.ipynb),
the graph from [network construction](../02_network_construction/NetworkConstruction.ipynb),
and the Intermediate embedding from [MeSH scope](../03_mesh_scope/MeSHScope.ipynb).
Only its drug IDs are reused; MeSH representations are fitted anew on development drugs.
Set `DATA_DIR`, `GRAPH_PATH`, `PRIVATE_PREPARED`, and `PRIVATE_RUN`
in the paths cell above.

Enable `PREPARE_SPLITS` to create `PRIVATE_PREPARED`. Enable
`RUN_TRAINING` to run all nine split–seed fits and export the public-style report.
Matching prepared splits and completed fits can be verified and reused.
Memberships, split embeddings/transforms, predictions, gates, caches, and
checkpoints are private outputs.

In [5]:
if PREPARE_SPLITS:
    from dtpkg.inductive.prepare_seen_unseen import prepare

    prepare(PRIVATE_PREPARED, data_dir=DATA_DIR, graph_path=GRAPH_PATH)

if RUN_TRAINING:
    from dtpkg.cli.run_inductive import main as run_inductive

    run_inductive([
        "--prepared-dir", str(PRIVATE_PREPARED), "--out-dir", str(PRIVATE_RUN),
        "--data-dir", str(DATA_DIR), "--graph-path", str(GRAPH_PATH),
        "--export-report",
    ])
    render_figures(PRIVATE_RUN / "figure_data", OUTPUT_DIR / "new_run")

## Interpretation and reproduction limits

The support-conditioned, balanced benchmark and three overlapping holdouts
limit generalization. Approximate corrected tests do not fully account for
shared-drug/network dependence. Different architectures, scaling, optimization,
and cohort composition also prevent attributing gains to gating alone. The
figure reporting choices were made after inspecting the results and remain
exploratory.

DrugBank-derived inputs and record-level outputs are not distributed. The public
aggregate tables suffice to redraw both figures. They preserve the completed
study run; packaging did not retrain it. See the
[snapshot provenance](tables/snapshot_provenance.json) for source hashes and
retained fields.